# Distance-Based PD Classification with MR-TAE-Fusion Denoising

## Pipeline
1. Generate/Load simulated distance-based signals (1.0mm, 1.8mm, 2.0mm, 2.5mm)
2. Apply MR-TAE-Fusion denoising
3. Train 4-class distance classifier
4. Evaluate on real-world data (data2.5mm.mat)
5. Compare with senior's MATLAB method

In [ ]:
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Add project to path
PROJECT_ROOT = Path(r'D:\New folder (2)\Data\OneDrive_2025-12-13\Try Model')
sys.path.insert(0, str(PROJECT_ROOT))

# Check device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Generate Training Data

In [ ]:
from mr_tae_fusion.data.distance_signals import (
    generate_batch, 
    DistanceSignalConfig,
    generate_distance_pd_signal,
    generate_noisy_signal
)

# Configuration
config = DistanceSignalConfig()

# Generate training data
print("Generating training data...")
train_data = generate_batch(
    batch_size=200,  # 200 samples per class = 800 total
    config=config,
    snr_range=(-10, 5),  # Mix of difficulty levels
    seed=42
)

# Generate validation data
print("Generating validation data...")
val_data = generate_batch(
    batch_size=50,  # 50 per class = 200 total
    config=config,
    snr_range=(-10, 5),
    seed=123
)

# Generate test data (harder - lower SNR)
print("Generating test data...")
test_data = generate_batch(
    batch_size=50,
    config=config,
    snr_range=(-15, 0),  # Harder test set
    seed=456
)

print(f"\nTraining: {len(train_data['labels'])} samples")
print(f"Validation: {len(val_data['labels'])} samples")
print(f"Test: {len(test_data['labels'])} samples")
print(f"\nClass distribution (train): {np.bincount(train_data['labels'])}")

In [ ]:
# Visualize sample signals
fig, axes = plt.subplots(4, 2, figsize=(14, 12))

class_names = ['1.0mm', '1.8mm', '2.0mm', '2.5mm']

for i in range(4):
    # Find first sample of each class
    idx = np.where(train_data['labels'] == i)[0][0]
    
    clean = train_data['clean'][idx]
    noisy = train_data['noisy'][idx]
    
    axes[i, 0].plot(clean, linewidth=0.8)
    axes[i, 0].set_title(f'{class_names[i]} - Clean')
    axes[i, 0].set_ylabel('Amplitude')
    
    axes[i, 1].plot(noisy, linewidth=0.8, color='orange')
    axes[i, 1].set_title(f'{class_names[i]} - Noisy (SNR: {train_data["snr"][idx]:.1f} dB)')

axes[-1, 0].set_xlabel('Sample')
axes[-1, 1].set_xlabel('Sample')
plt.tight_layout()
plt.suptitle('Simulated Distance-Based PD Signals', y=1.02, fontsize=14, fontweight='bold')
plt.show()

## 2. Create DataLoaders

In [ ]:
def create_dataloaders(data_dict, batch_size=32, shuffle=True):
    """Create PyTorch DataLoader from data dictionary."""
    # Use noisy signals as input (denoising will happen in model or separately)
    X = torch.tensor(data_dict['noisy'], dtype=torch.float32).unsqueeze(1)  # (N, 1, L)
    X_clean = torch.tensor(data_dict['clean'], dtype=torch.float32).unsqueeze(1)
    y = torch.tensor(data_dict['labels'], dtype=torch.long)
    
    dataset = TensorDataset(X, X_clean, y)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)
    
    return loader

# Create dataloaders
train_loader = create_dataloaders(train_data, batch_size=32, shuffle=True)
val_loader = create_dataloaders(val_data, batch_size=32, shuffle=False)
test_loader = create_dataloaders(test_data, batch_size=32, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

## 3. Create Distance Classifier

In [ ]:
from mr_tae_fusion.models.distance_classifier import DistanceClassifier

# Create classifier
classifier = DistanceClassifier(
    in_channels=1,
    signal_length=2001,
    num_classes=4,
    hidden_channels=[32, 64, 128]
).to(device)

# Count parameters
total_params = sum(p.numel() for p in classifier.parameters())
print(f"Total parameters: {total_params:,}")

# Test forward pass
with torch.no_grad():
    dummy = torch.randn(4, 1, 2001).to(device)
    out = classifier(dummy)
    print(f"Input shape: {dummy.shape}")
    print(f"Output shape: {out.shape}")

## 4. Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for noisy, clean, labels in loader:
        # Use clean signals for initial training (no denoising needed yet)
        X = clean.to(device)
        y = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += y.size(0)
        correct += predicted.eq(y).sum().item()
    
    return total_loss / len(loader), 100. * correct / total


def evaluate(model, loader, criterion, device, use_noisy=False):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for noisy, clean, labels in loader:
            X = noisy.to(device) if use_noisy else clean.to(device)
            y = labels.to(device)
            
            outputs = model(X)
            loss = criterion(outputs, y)
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += y.size(0)
            correct += predicted.eq(y).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    
    return total_loss / len(loader), 100. * correct / total, all_preds, all_labels

In [ ]:
# Training configuration
NUM_EPOCHS = 50
LEARNING_RATE = 1e-3

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(classifier.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, NUM_EPOCHS)

# Training history
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': []
}

best_val_acc = 0
best_model_state = None

print("Starting training...")
print("=" * 60)

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(classifier, train_loader, optimizer, criterion, device)
    val_loss, val_acc, _, _ = evaluate(classifier, val_loader, criterion, device)
    
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = classifier.state_dict().copy()
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

print("=" * 60)
print(f"Best Validation Accuracy: {best_val_acc:.2f}%")

# Load best model
classifier.load_state_dict(best_model_state)

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0].plot(history['val_loss'], label='Validation', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_acc'], label='Train', linewidth=2)
axes[1].plot(history['val_acc'], label='Validation', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Evaluation on Clean vs Noisy Signals

In [ ]:
# Evaluate on clean signals (best case)
_, clean_acc, clean_preds, clean_labels = evaluate(
    classifier, test_loader, criterion, device, use_noisy=False
)

# Evaluate on noisy signals (without denoising)
_, noisy_acc, noisy_preds, noisy_labels = evaluate(
    classifier, test_loader, criterion, device, use_noisy=True
)

print("="*60)
print("TEST RESULTS (Simulated Data)")
print("="*60)
print(f"Accuracy on CLEAN signals: {clean_acc:.2f}%")
print(f"Accuracy on NOISY signals: {noisy_acc:.2f}%")
print(f"Performance drop: {clean_acc - noisy_acc:.2f}%")
print("="*60)

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
class_names = ['1.0mm', '1.8mm', '2.0mm', '2.5mm']

# Clean signals
cm_clean = confusion_matrix(clean_labels, clean_preds)
sns.heatmap(cm_clean, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=class_names, yticklabels=class_names)
axes[0].set_title(f'Clean Signals\nAccuracy: {clean_acc:.2f}%')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Noisy signals
cm_noisy = confusion_matrix(noisy_labels, noisy_preds)
sns.heatmap(cm_noisy, annot=True, fmt='d', cmap='Oranges', ax=axes[1],
            xticklabels=class_names, yticklabels=class_names)
axes[1].set_title(f'Noisy Signals (No Denoising)\nAccuracy: {noisy_acc:.2f}%')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

print("\nClassification Report (Noisy Signals):")
print(classification_report(noisy_labels, noisy_preds, target_names=class_names))

## 6. Load and Test on Real-World Data

In [ ]:
# Load real-world data from Q.lin
def load_mat_file(filepath):
    """Load .mat file."""
    try:
        import scipy.io as sio
        return sio.loadmat(str(filepath))
    except NotImplementedError:
        pass
    
    try:
        import mat73
        return mat73.loadmat(str(filepath))
    except:
        pass
    
    import h5py
    data = {}
    with h5py.File(str(filepath), 'r') as f:
        for key in f.keys():
            data[key] = np.array(f[key])
    return data

# Load Q.lin datasets
DATA_DIR = Path(r'D:\New folder (2)\Data\OneDrive_2025-12-13\Datset from Q.lin')

real_data = {}
files = [
    ('1.0mm', 'data1.0mm.mat'),
    ('1.8mm', 'data1.8mm.mat'),
    ('2.0mm', 'data2.0mm.mat'),
    ('2.5mm', 'data2.5mm.mat')
]

for label, filename in files:
    filepath = DATA_DIR / filename
    if filepath.exists():
        print(f"Loading {filename}...")
        try:
            mat = load_mat_file(filepath)
            # Find main data variable
            for key, value in mat.items():
                if not key.startswith('__') and isinstance(value, np.ndarray):
                    real_data[label] = value
                    print(f"  {label}: shape={value.shape}")
                    break
        except Exception as e:
            print(f"  Error: {e}")

print(f"\nLoaded {len(real_data)} datasets")

In [ ]:
# Prepare real-world data for testing
def preprocess_real_data(data, target_length=2001):
    """Preprocess real data to match model input requirements."""
    if data.ndim == 1:
        data = data.reshape(1, -1)
    
    # Ensure shape is (n_samples, signal_length)
    if data.shape[0] > data.shape[1]:
        data = data.T
    
    n_samples, signal_length = data.shape
    
    processed = []
    for i in range(n_samples):
        signal = data[i]
        
        if len(signal) >= target_length:
            # Take first target_length samples
            processed.append(signal[:target_length])
        else:
            # Pad with zeros
            padded = np.zeros(target_length)
            padded[:len(signal)] = signal
            processed.append(padded)
    
    processed = np.array(processed)
    
    # Normalize
    for i in range(len(processed)):
        max_val = np.max(np.abs(processed[i]))
        if max_val > 0:
            processed[i] = processed[i] / max_val
    
    return processed

# Process all real datasets
real_processed = {}
for label, data in real_data.items():
    real_processed[label] = preprocess_real_data(data, target_length=2001)
    print(f"{label}: {real_processed[label].shape}")

In [ ]:
# Test classifier on real-world data
classifier.eval()

class_to_idx = {'1.0mm': 0, '1.8mm': 1, '2.0mm': 2, '2.5mm': 3}
class_names = ['1.0mm', '1.8mm', '2.0mm', '2.5mm']

all_preds = []
all_labels = []
per_class_acc = {}

print("\n" + "="*60)
print("REAL-WORLD DATA EVALUATION")
print("="*60)

for label, data in real_processed.items():
    # Convert to tensor
    X = torch.tensor(data, dtype=torch.float32).unsqueeze(1).to(device)
    true_class = class_to_idx[label]
    
    # Predict
    with torch.no_grad():
        preds = classifier.predict(X).cpu().numpy()
    
    # Calculate accuracy for this class
    correct = np.sum(preds == true_class)
    total = len(preds)
    acc = 100.0 * correct / total
    per_class_acc[label] = acc
    
    # Store for overall metrics
    all_preds.extend(preds)
    all_labels.extend([true_class] * total)
    
    # Print distribution of predictions
    pred_counts = np.bincount(preds, minlength=4)
    print(f"\n{label} (n={total}):")
    print(f"  Accuracy: {acc:.2f}%")
    print(f"  Predictions: 1.0mm={pred_counts[0]}, 1.8mm={pred_counts[1]}, "
          f"2.0mm={pred_counts[2]}, 2.5mm={pred_counts[3]}")

# Overall accuracy
overall_acc = 100.0 * np.sum(np.array(all_preds) == np.array(all_labels)) / len(all_labels)
print("\n" + "="*60)
print(f"OVERALL ACCURACY: {overall_acc:.2f}%")
print("="*60)

In [ ]:
# Confusion matrix for real-world data
if len(all_preds) > 0:
    cm_real = confusion_matrix(all_labels, all_preds)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_real, annot=True, fmt='d', cmap='Greens',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Real-World Data Classification\nOverall Accuracy: {overall_acc:.2f}%')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()
    
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=class_names))

## 7. Save Model

In [ ]:
# Save trained classifier
save_path = PROJECT_ROOT / 'mr_tae_fusion' / 'checkpoints'
save_path.mkdir(parents=True, exist_ok=True)

torch.save({
    'model_state_dict': classifier.state_dict(),
    'history': history,
    'config': {
        'signal_length': 2001,
        'num_classes': 4,
        'class_names': class_names
    }
}, save_path / 'distance_classifier.pth')

print(f"Model saved to {save_path / 'distance_classifier.pth'}")

## 8. Summary & Recommendations

In [ ]:
print("\n" + "="*70)
print("SUMMARY")
print("="*70)

print("\n1. SIMULATED DATA RESULTS:")
print(f"   - Clean signal accuracy: {clean_acc:.2f}%")
print(f"   - Noisy signal accuracy: {noisy_acc:.2f}%")
print(f"   - Performance drop without denoising: {clean_acc - noisy_acc:.2f}%")

print("\n2. REAL-WORLD DATA RESULTS:")
print(f"   - Overall accuracy: {overall_acc:.2f}%")
for label, acc in per_class_acc.items():
    print(f"   - {label}: {acc:.2f}%")

print("\n3. RECOMMENDATIONS:")
print("   - Apply MR-TAE-Fusion denoising before classification to improve noisy performance")
print("   - The 2.5mm data should be used as test set (real-world)")
print("   - If accuracy on noisy data is low, integrate denoising pipeline")

print("\n4. NEXT STEPS:")
print("   - Load pretrained MR-TAE-Fusion denoiser")
print("   - Apply denoising before classification")
print("   - Compare results with/without denoising")
print("   - Benchmark against senior's MATLAB method")

print("\n" + "="*70)